In [ ]:
import os
import copy
import subprocess
from glob import glob
from itertools import product
from datetime import datetime
from pathlib import Path
from string import Template
from utils.notebook import isnotebook
if isnotebook():
    home_dir = os.path.expanduser("~")
    os.chdir(os.path.join(home_dir, "aiwq"))

    # Autoreload modified packages
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

# Inline plotting setup
%matplotlib inline
%config InlineBackend.figure_formats = ['pdf', 'svg']
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Markdown, display
from AI_WQ_package import forecast_submission
from src.utils.data_io import *
from src.viz.viz_utils_pbc import *
from models.utils.general_util import printf
from models.utils.eval_util import get_target_dates
from models.utils.data_utils import get_measurement_variable
from models.utils.models_util import get_submodel_name, get_selected_submodel_name
from utils.timing import tic, toc
from utils.data_io import save_to_netcdf, load_data
from utils.logging import printf


#### December 2025 cold air outbreak in the Eastern United States

In [ ]:
if True:
    fig_gt_id = 'era5-tas'
    fig_horizon = 26
    fig_target_date = None
    fig_issuance_date = '20251120'
    fig_team_name = 'Dynamical_S2SDatabase'
    fig_model_name = 'ECMWF'
    fig_bbox_name = 'us'
    fig_quintile = 0.2
    fig_y_suptitle = 0.89
    
    
    plot_probability_maps(
        gt_id = fig_gt_id,
        horizon = fig_horizon,
        target_date = fig_target_date,
        issuance_date = fig_issuance_date,
        team_name = fig_team_name,
        model_name = fig_model_name,
        bbox_name = fig_bbox_name, 
        quintile = fig_quintile,
        y_suptitle = fig_y_suptitle,
    )

#### ECMWF, Debiased ECMWF, PBC-ECMWF extreme barplots (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False
fig_prefixes = ["F95_","F5_"]
fig_horizons=["19", "26"]
fig_gt_ids = []
for prefix in fig_prefixes:
    for var in ["tas", "pr", "mslp"]:
        fig_gt_ids.append(f"era5-{prefix}{var}")

tic()
all_wtd_mse = get_all_metrics(
    model_names=fig_model_names,
    model_names_str=fig_model_names_str,
    metrics=['wtd_mse'],
    gt_ids=fig_gt_ids,
    horizons=fig_horizons,
    target_dates_list=fig_target_dates_list,
    common_dates=True,
    verbose=fig_verbose)
# Index all_wtd_mse by task alone
all_wtd_mse = {t: all_wtd_mse[(m, t, d)] for m, t, d in all_wtd_mse.keys()}
toc()
for horizon in fig_horizons:
    fig_show=True
    fig_save=True
    plot_single_horizon_bss_barplot(all_wtd_mse,
                    horizon = horizon,
                    model_names=fig_model_names,
                    target_dates=fig_target_dates,
                    show_fig=fig_show,
                    save_fig=fig_save,
                    prefixes=fig_prefixes,
                    verbose=fig_verbose,
                    y_bottom=-0.113 if horizon=="19" else -0.145,)

if False:
    print_improvements(all_wtd_mse, 
                        model_name='pbc_ecmwf_combo', 
                        baseline_models=['ecmwf', 'debiased_ecmwf'])


#### ECMWF, Debiased ECMWF, PBC-ECMWF extreme map plots (2016-2024)

In [ ]:
fig_model_names=['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_target_dates="std_test"
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
figure_show = True
figure_save = True
prefixes = ["F95_","F5_"]
for horizon in [19, 26]:
    plot_single_horizon_bss_diff_grid_6x4(
        horizon = horizon, 
        prefixes = prefixes,
        model_names=fig_model_names, 
        target_dates=fig_target_dates,
        diff_cmap=diff_cmap, skill_cmap=skill_cmap, 
        show_fig=figure_show, save_fig=figure_save)